# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [22]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Define the tickers: AAPL, SPY, TSLA (required) + QQQ (starts with Q), MSFT, NVDA
tickers = ['AAPL', 'SPY', 'TSLA', 'QQQ', 'MSFT', 'NVDA']

# Date range: September 2024 to August 2026 (24 months)
# The last month that ended before September 2026 is August 2026
start_date = '2024-09-01'
end_date = '2026-08-31'

# Download data for all tickers
print("Downloading historical data...")
data = yf.download(tickers, start=start_date, end=end_date, progress=False)

# Get the adjusted close prices (returned as 'Close' in recent yfinance versions)
# The data has MultiIndex columns with ('Close', 'AAPL'), etc.
close_data = data['Close']

# Get the last trading day of each month
print("Processing data to get last trading day of each month...")
monthly_data = close_data.resample('ME').last()

# Reset index to make date a column
df = monthly_data.reset_index()

# Rename the 'Date' column to 'date'
df = df.rename(columns={'Date': 'date'})

# Format the date column as YYYY-MM-DD
df['date'] = df['date'].dt.strftime('%Y-%m-%d')

# Reorder columns: date first, then tickers
columns_to_keep = ['date'] + tickers
df = df[columns_to_keep]

print(f"\nData shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nLast few rows:")
print(df.tail())

# Check for any missing values
print(f"\nMissing values per column:")
print(df.isnull().sum())


Processing data to get last trading day of each month...

Data shape: (24, 7)

First few rows:
Ticker        date        AAPL         SPY        TSLA         QQQ  \
0       2024-09-30  231.067078  562.210510  261.630005  483.683289   
1       2024-10-31  224.035919  557.193481  249.850006  479.501282   
2       2024-11-30  235.620117  590.420898  345.160004  505.158508   
3       2024-12-31  248.615784  576.215332  403.839996  507.452179   
4       2025-01-31  234.299683  591.690308  404.600006  518.430420   

Ticker        MSFT        NVDA  
0       423.608215  121.250549  
1       400.030731  132.552872  
2       417.709106  138.034332  
3       415.775696  134.089737  
4       409.423157  119.890938  

Last few rows:
Ticker        date        AAPL         SPY        TSLA         QQQ  \
19      2026-04-30  270.866638  716.813293  381.630005  667.006958   
20      2026-05-31  311.791107  754.536133  435.790009  737.499512   
21      2026-06-30  289.110657  746.770020  420.600006  736.

Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [23]:
# YOUR CHANGES HERE

# Save to TSV file
output_path = 'historical_prices.tsv'
df.to_csv(output_path, sep='\t', index=False)
print(f"\nData saved to {output_path}")

# Display the dataframe
df


Data saved to historical_prices.tsv


Ticker,date,AAPL,SPY,TSLA,QQQ,MSFT,NVDA
0,2024-09-30,231.067078,562.210510,261.630005,483.683289,423.608215,121.250549
1,2024-10-31,224.035919,557.193481,249.850006,479.501282,400.030731,132.552872
2,2024-11-30,235.620117,590.420898,345.160004,505.158508,417.709106,138.034332
3,2024-12-31,248.615784,576.215332,403.839996,507.452179,415.775696,134.089737
4,2025-01-31,234.299683,591.690308,404.600006,518.430420,409.423157,119.890938
5,2025-02-28,240.361588,584.178955,292.980011,504.414764,392.383728,124.733696
6,2025-03-31,220.772079,551.628967,259.160004,466.148895,371.034393,108.228325
7,2025-04-30,211.200958,546.846191,282.160004,472.660156,390.673859,108.767563
8,2025-05-31,199.883957,581.212830,346.459991,516.042297,455.853821,134.940903
9,2025-06-30,204.183167,611.079041,317.660004,548.995911,492.541168,157.779861


Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [24]:
# Load the historical prices data
prices_df = pd.read_csv('historical_prices.tsv', sep='\t')

# Separate date and price columns
dates = prices_df['date']
prices_only = prices_df[tickers]

# Calculate monthly returns using pct_change()
# This calculates (current - previous) / previous for each row
returns = prices_only.pct_change()

# Drop the first row (which is NaN since there's no previous price)
returns_clean = returns.iloc[1:].reset_index(drop=True)

# Add the date column back (starting from the second date)
returns_with_date = pd.DataFrame({
    'date': dates.iloc[1:].reset_index(drop=True),
    **{ticker: returns_clean[ticker] for ticker in tickers}
})

print(f"Returns data shape: {returns_with_date.shape}")
print(f"\nFirst few rows:")
print(returns_with_date.head())
print(f"\nLast few rows:")
print(returns_with_date.tail())

# Check for any missing values
print(f"\nMissing values per column:")
print(returns_with_date.isnull().sum())

Returns data shape: (23, 7)

First few rows:
         date      AAPL       SPY      TSLA       QQQ      MSFT      NVDA
0  2024-10-31 -0.030429 -0.008924 -0.045025 -0.008646 -0.055659  0.093215
1  2024-11-30  0.051707  0.059634  0.381469  0.053508  0.044193  0.041353
2  2024-12-31  0.055155 -0.024060  0.170008  0.004540 -0.004629 -0.028577
3  2025-01-31 -0.057583  0.026856  0.001882  0.021634 -0.015279 -0.105890
4  2025-02-28  0.025872 -0.012695 -0.275877 -0.027035 -0.041618  0.040393

Last few rows:
          date      AAPL       SPY      TSLA       QQQ      MSFT      NVDA
18  2026-04-30  0.069191  0.105053  0.026577  0.156901  0.101602  0.144323
19  2026-05-31  0.151087  0.052626  0.141918  0.105685  0.106516  0.057975
20  2026-06-30 -0.072742 -0.010293 -0.034856 -0.001491 -0.171509 -0.051230
21  2026-07-31  0.067563  0.000348 -0.260081 -0.065739  0.245831  0.003299
22  2026-08-31  0.035822  0.029878  0.120626  0.041338  0.107111  0.083686

Missing values per column:
date    0
AAPL   

Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [25]:
# YOUR CHANGES HERE

# Save to TSV file
returns_output_path = 'historical_returns.tsv'
returns_with_date.to_csv(returns_output_path, sep='\t', index=False)
print(f"\nReturns saved to {returns_output_path}")

# Display the dataframe
returns_with_date


Returns saved to historical_returns.tsv


,date,AAPL,SPY,TSLA,QQQ,MSFT,NVDA
0,2024-10-31,-0.030429,-0.008924,-0.045025,-0.008646,-0.055659,0.093215
1,2024-11-30,0.051707,0.059634,0.381469,0.053508,0.044193,0.041353
2,2024-12-31,0.055155,-0.024060,0.170008,0.004540,-0.004629,-0.028577
3,2025-01-31,-0.057583,0.026856,0.001882,0.021634,-0.015279,-0.105890
4,2025-02-28,0.025872,-0.012695,-0.275877,-0.027035,-0.041618,0.040393
5,2025-03-31,-0.081500,-0.055719,-0.115435,-0.075862,-0.054409,-0.132325
6,2025-04-30,-0.043353,-0.008670,0.088748,0.013968,0.052932,0.004982
7,2025-05-31,-0.053584,0.062845,0.227885,0.091783,0.166840,0.240636
8,2025-06-30,0.021509,0.051386,-0.083126,0.063858,0.080481,0.169252
9,2025-07-31,0.011698,0.023032,-0.029560,0.024237,0.072556,0.125831


Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [26]:
# Compute the mean (average) return for each asset
# This is the expected return estimate for each asset
mean_returns = returns_with_date[tickers].mean()

print("Estimated Expected Returns for Each Asset:")
print("=" * 40)
for ticker in tickers:
    print(f"{ticker}: {mean_returns[ticker]:.6f} ({mean_returns[ticker]*100:.4f}%)")
print("=" * 40)

# Display as a pandas Series for inspection
mean_returns

Estimated Expected Returns for Each Asset:
AAPL: 0.016097 (1.6097%)
SPY: 0.014380 (1.4380%)
TSLA: 0.024361 (2.4361%)
QQQ: 0.018607 (1.8607%)
MSFT: 0.012617 (1.2617%)
NVDA: 0.029858 (2.9858%)


AAPL    0.016097
SPY     0.014380
TSLA    0.024361
QQQ     0.018607
MSFT    0.012617
NVDA    0.029858
dtype: float64

Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [27]:
# Create a DataFrame with asset names and their estimated returns
estimated_returns_df = pd.DataFrame({
    'asset': tickers,
    'estimated_return': [mean_returns[ticker] for ticker in tickers]
})

# Save to TSV file
estimated_returns_path = 'estimated_returns.tsv'
estimated_returns_df.to_csv(estimated_returns_path, sep='\t', index=False)
print(f"Estimated returns saved to {estimated_returns_path}\n")

# Display the dataframe
estimated_returns_df

Estimated returns saved to estimated_returns.tsv



,asset,estimated_return
0,AAPL,0.016097
1,SPY,0.014380
2,TSLA,0.024361
3,QQQ,0.018607
4,MSFT,0.012617
5,NVDA,0.029858


Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [28]:
# Calculate the covariance matrix for the asset returns
# Use only the return columns (exclude date)
covariance_matrix = returns_with_date[tickers].cov()

print("Estimated Covariance Matrix for Asset Returns:")
print("=" * 60)
print(covariance_matrix)
print("=" * 60)
print("\nCovariance matrix interpretation:")
print("- Diagonal elements: variance of each asset")
print("- Off-diagonal elements: covariance between assets")
print("- Positive covariance: assets tend to move together")
print("- Negative covariance: assets tend to move opposite")
print("- Zero covariance: assets move independently\n")

# Display as a pandas DataFrame for inspection
covariance_matrix

Estimated Covariance Matrix for Asset Returns:
          AAPL       SPY      TSLA       QQQ      MSFT      NVDA
AAPL  0.004027  0.001061  0.002839  0.001391  0.002342  0.001121
SPY   0.001061  0.001384  0.002796  0.001859  0.001864  0.002277
TSLA  0.002839  0.002796  0.025419  0.005002  0.002540  0.004240
QQQ   0.001391  0.001859  0.005002  0.002984  0.002092  0.003425
MSFT  0.002342  0.001864  0.002540  0.002092  0.009086  0.004995
NVDA  0.001121  0.002277  0.004240  0.003425  0.004995  0.008864

Covariance matrix interpretation:
- Diagonal elements: variance of each asset
- Off-diagonal elements: covariance between assets
- Positive covariance: assets tend to move together
- Negative covariance: assets tend to move opposite
- Zero covariance: assets move independently



,AAPL,SPY,TSLA,QQQ,MSFT,NVDA
AAPL,0.004027,0.001061,0.002839,0.001391,0.002342,0.001121
SPY,0.001061,0.001384,0.002796,0.001859,0.001864,0.002277
TSLA,0.002839,0.002796,0.025419,0.005002,0.002540,0.004240
QQQ,0.001391,0.001859,0.005002,0.002984,0.002092,0.003425
MSFT,0.002342,0.001864,0.002540,0.002092,0.009086,0.004995
NVDA,0.001121,0.002277,0.004240,0.003425,0.004995,0.008864


Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [29]:
# Save the covariance matrix to a TSV file
covariance_path = 'estimated_covariance.tsv'
covariance_matrix.to_csv(covariance_path, sep='\t')
print(f"Covariance matrix saved to {covariance_path}\n")

# Display the dataframe
covariance_matrix

Covariance matrix saved to estimated_covariance.tsv



,AAPL,SPY,TSLA,QQQ,MSFT,NVDA
AAPL,0.004027,0.001061,0.002839,0.001391,0.002342,0.001121
SPY,0.001061,0.001384,0.002796,0.001859,0.001864,0.002277
TSLA,0.002839,0.002796,0.025419,0.005002,0.002540,0.004240
QQQ,0.001391,0.001859,0.005002,0.002984,0.002092,0.003425
MSFT,0.002342,0.001864,0.002540,0.002092,0.009086,0.004995
NVDA,0.001121,0.002277,0.004240,0.003425,0.004995,0.008864


Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [30]:
import numpy as np

# Find the asset with the maximum expected return
max_return_idx = mean_returns.idxmax()
max_return_value = mean_returns.max()

print("Maximum Return Portfolio Analysis:")
print("=" * 50)
print(f"Asset with highest expected return: {max_return_idx}")
print(f"Expected return: {max_return_value:.6f} ({max_return_value*100:.4f}%)")
print("=" * 50)

# Create the maximum return portfolio
# Allocate 100% to the asset with highest return, 0% to others
max_return_portfolio = pd.Series(0.0, index=tickers)
max_return_portfolio[max_return_idx] = 1.0

# Calculate portfolio metrics
portfolio_return = (max_return_portfolio * mean_returns).sum()
portfolio_variance = max_return_portfolio.T @ covariance_matrix @ max_return_portfolio
portfolio_std = np.sqrt(portfolio_variance)

print(f"\nMaximum Return Portfolio:")
print(f"Expected Return: {portfolio_return:.6f} ({portfolio_return*100:.4f}%)")
print(f"Standard Deviation (Risk): {portfolio_std:.6f} ({portfolio_std*100:.4f}%)")
print("\nPortfolio Allocations:")
print(max_return_portfolio)


Maximum Return Portfolio Analysis:
Asset with highest expected return: NVDA
Expected return: 0.029858 (2.9858%)

Maximum Return Portfolio:
Expected Return: 0.029858 (2.9858%)
Standard Deviation (Risk): 0.094146 (9.4146%)

Portfolio Allocations:
AAPL    0.0
SPY     0.0
TSLA    0.0
QQQ     0.0
MSFT    0.0
NVDA    1.0
dtype: float64


Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [31]:
# Save the maximum return portfolio to a TSV file
max_return_df = pd.DataFrame({
    'asset': tickers,
    'allocation': max_return_portfolio.values
})

max_return_path = 'maximum_return.tsv'
max_return_df.to_csv(max_return_path, sep='\t', index=False)
print(f"Maximum return portfolio saved to {max_return_path}\n")

# Verify allocations sum to 1
print(f"Sum of allocations: {max_return_df['allocation'].sum()}")
print("\nMaximum Return Portfolio:")
print(max_return_df)


Maximum return portfolio saved to maximum_return.tsv

Sum of allocations: 1.0

Maximum Return Portfolio:
  asset  allocation
0  AAPL         0.0
1   SPY         0.0
2  TSLA         0.0
3   QQQ         0.0
4  MSFT         0.0
5  NVDA         1.0


Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [32]:
from scipy.optimize import minimize

# Objective function: portfolio variance
def portfolio_variance(weights, cov_matrix):
    return weights.T @ cov_matrix @ weights

# Constraint: weights sum to 1
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

# Bounds: weights between 0 and 1 (long-only portfolio)
bounds = tuple((0, 1) for _ in range(len(tickers)))

# Initial guess: equal weights
x0 = np.array([1/len(tickers)] * len(tickers))

# Solve for minimum variance portfolio
result = minimize(portfolio_variance, x0, args=(covariance_matrix.values,),
                 method='SLSQP', bounds=bounds, constraints=constraints)

min_risk_portfolio = pd.Series(result.x, index=tickers)

# Calculate portfolio metrics
min_portfolio_variance = portfolio_variance(min_risk_portfolio.values, covariance_matrix.values)
min_portfolio_std = np.sqrt(min_portfolio_variance)
min_portfolio_return = (min_risk_portfolio * mean_returns).sum()

print("Minimum Risk Portfolio Analysis:")
print("=" * 50)
print(f"Portfolio Allocations:")
for ticker in tickers:
    print(f"  {ticker}: {min_risk_portfolio[ticker]:.6f} ({min_risk_portfolio[ticker]*100:.4f}%)")
print("=" * 50)
print(f"\nPortfolio Metrics:")
print(f"Expected Return: {min_portfolio_return:.6f} ({min_portfolio_return*100:.4f}%)")
print(f"Standard Deviation (Risk): {min_portfolio_std:.6f} ({min_portfolio_std*100:.4f}%)")
print(f"\nSum of allocations: {min_risk_portfolio.sum():.10f}")


Minimum Risk Portfolio Analysis:
Portfolio Allocations:
  AAPL: 0.098063 (9.8063%)
  SPY: 0.901937 (90.1937%)
  TSLA: 0.000000 (0.0000%)
  QQQ: 0.000000 (0.0000%)
  MSFT: 0.000000 (0.0000%)
  NVDA: 0.000000 (0.0000%)

Portfolio Metrics:
Expected Return: 0.014548 (1.4548%)
Standard Deviation (Risk): 0.036769 (3.6769%)

Sum of allocations: 1.0000000000


Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [33]:
# Save the minimum risk portfolio to a TSV file
min_risk_df = pd.DataFrame({
    'asset': tickers,
    'allocation': min_risk_portfolio.values
})

min_risk_path = 'minimum_risk.tsv'
min_risk_df.to_csv(min_risk_path, sep='\t', index=False)
print(f"Minimum risk portfolio saved to {min_risk_path}\n")

# Verify allocations sum to 1
print(f"Sum of allocations: {min_risk_df['allocation'].sum()}")
print("\nMinimum Risk Portfolio:")
print(min_risk_df)


Minimum risk portfolio saved to minimum_risk.tsv

Sum of allocations: 1.0

Minimum Risk Portfolio:
  asset    allocation
0  AAPL  9.806349e-02
1   SPY  9.019365e-01
2  TSLA  0.000000e+00
3   QQQ  2.625095e-17
4  MSFT  2.425352e-18
5  NVDA  0.000000e+00


Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [34]:
# Create 101 target returns from minimum risk to maximum return
min_risk_return = min_portfolio_return
max_return = portfolio_return
target_returns = np.linspace(min_risk_return, max_return, 101)

# Objective function: minimize portfolio variance
def portfolio_variance(weights, cov_matrix):
    return weights.T @ cov_matrix @ weights

# Helper function to create a return constraint with proper closure
def make_return_constraint(target_ret, mean_rets):
    def return_constraint(w):
        return np.dot(w, mean_rets) - target_ret
    return return_constraint

# Store results for all portfolios
efficient_portfolios = []

# For each target return, find the minimum variance portfolio
for idx, target_return in enumerate(target_returns):
    # Constraints: weights sum to 1 and portfolio return equals target
    constraints = (
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': make_return_constraint(target_return, mean_returns.values)}
    )
    
    # Bounds: weights between 0 and 1 (long-only)
    bounds = tuple((0, 1) for _ in range(len(tickers)))
    
    # Initial guess: equal weights
    x0 = np.array([1/len(tickers)] * len(tickers))
    
    # Solve the optimization problem
    result = minimize(portfolio_variance, x0, args=(covariance_matrix.values,),
                     method='SLSQP', bounds=bounds, constraints=constraints)
    
    if result.success:
        # Get the optimal weights
        weights = result.x
        
        # Calculate portfolio metrics
        portfolio_std = np.sqrt(portfolio_variance(weights, covariance_matrix.values))
        
        # Store portfolio data
        portfolio_data = {
            'index': idx,
            'return': target_return,
            'risk': portfolio_std
        }
        
        # Add allocations for each ticker
        for ticker, weight in zip(tickers, weights):
            portfolio_data[ticker] = weight
        
        efficient_portfolios.append(portfolio_data)
    else:
        print(f"Optimization failed for portfolio {idx} with target return {target_return}")

# Create dataframe
efficient_frontier_df = pd.DataFrame(efficient_portfolios)

# Verify some portfolios
print("Efficient Frontier Portfolios:")
print("=" * 80)
print(f"\nFirst portfolio (minimum risk):")
print(efficient_frontier_df.iloc[0])
print(f"\nLast portfolio (maximum return):")
print(efficient_frontier_df.iloc[-1])
print(f"\nTotal portfolios: {len(efficient_frontier_df)}")
print(f"\nDataframe shape: {efficient_frontier_df.shape}")
print(f"\nFirst few rows:")
print(efficient_frontier_df.head())
print(f"\nVerification - Portfolio returns match targets:")
print(f"First 5 portfolios:")
for i in range(min(5, len(efficient_frontier_df))):
    row = efficient_frontier_df.iloc[i]
    calculated_return = sum(row[ticker] * mean_returns[ticker] for ticker in tickers)
    print(f"  Portfolio {i}: Target return = {row['return']:.6f}, Calculated return = {calculated_return:.6f}")

Efficient Frontier Portfolios:

First portfolio (minimum risk):
index     0.000000e+00
return    1.454834e-02
risk      3.676879e-02
AAPL      9.806348e-02
SPY       9.019365e-01
TSLA      0.000000e+00
QQQ       1.646235e-17
MSFT      4.163336e-17
NVDA      0.000000e+00
Name: 0, dtype: float64

Last portfolio (maximum return):
index     1.000000e+02
return    2.985849e-02
risk      9.414628e-02
AAPL      0.000000e+00
SPY       1.026956e-15
TSLA      1.998252e-08
QQQ       6.661338e-16
MSFT      0.000000e+00
NVDA      1.000000e+00
Name: 100, dtype: float64

Total portfolios: 101

Dataframe shape: (101, 9)

First few rows:
   index    return      risk      AAPL       SPY          TSLA           QQQ  \
0      0  0.014548  0.036769  0.098063  0.901937  0.000000e+00  1.646235e-17   
1      1  0.014701  0.037066  0.158310  0.830963  2.317482e-18  1.034645e-02   
2      2  0.014855  0.038059  0.266630  0.729946  1.145189e-18  3.229047e-03   
3      3  0.015008  0.038035  0.250361  0.736866  4

Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [35]:
# Save the efficient frontier to a TSV file
efficient_frontier_path = 'efficient_frontier.tsv'
efficient_frontier_df.to_csv(efficient_frontier_path, sep='\t', index=False)
print(f"Efficient frontier portfolios saved to {efficient_frontier_path}\n")

# Display the dataframe
print("Efficient Frontier DataFrame:")
print(efficient_frontier_df)

Efficient frontier portfolios saved to efficient_frontier.tsv

Efficient Frontier DataFrame:
     index    return      risk          AAPL           SPY          TSLA  \
0        0  0.014548  0.036769  9.806348e-02  9.019365e-01  0.000000e+00   
1        1  0.014701  0.037066  1.583100e-01  8.309630e-01  2.317482e-18   
2        2  0.014855  0.038059  2.666296e-01  7.299459e-01  1.145189e-18   
3        3  0.015008  0.038035  2.503606e-01  7.368656e-01  4.770490e-18   
4        4  0.015161  0.038143  2.406057e-01  7.356469e-01  0.000000e+00   
..     ...       ...       ...           ...           ...           ...   
96      96  0.029246  0.090019  1.890801e-02  0.000000e+00  6.406646e-02   
97      97  0.029399  0.090945  4.163336e-16  0.000000e+00  8.354606e-02   
98      98  0.029552  0.091807  0.000000e+00  2.775558e-16  5.569738e-02   
99      99  0.029705  0.092876  0.000000e+00  6.106227e-16  2.784870e-02   
100    100  0.029858  0.094146  0.000000e+00  1.026956e-15  1.998252e-0

Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [36]:
# Repeat 1000 times to check stability of maximum return portfolio
num_resamples = 1000
highest_return_counts = {ticker: 0 for ticker in tickers}

# Set random seed for reproducibility (optional)
np.random.seed(42)

for resample_num in range(num_resamples):
    # Step 1: Generate 23 return samples using multivariate normal distribution
    # with the estimated mean returns and covariance matrix
    resampled_returns = np.random.multivariate_normal(
        mean=mean_returns.values,
        cov=covariance_matrix.values,
        size=23
    )
    
    # Step 2: Estimate the return of each asset from the resampled history
    # Calculate the mean return for each asset from the 23 samples
    estimated_returns_resampled = resampled_returns.mean(axis=0)
    
    # Step 3: Find which asset had the highest return in the resampled estimates
    highest_return_idx = np.argmax(estimated_returns_resampled)
    highest_return_ticker = tickers[highest_return_idx]
    
    # Count this occurrence
    highest_return_counts[highest_return_ticker] += 1

# Calculate probabilities
asset_probabilities = []
for ticker in tickers:
    probability = highest_return_counts[ticker] / num_resamples
    asset_probabilities.append({
        'asset': ticker,
        'probability': probability
    })

# Create dataframe
max_return_prob_df = pd.DataFrame(asset_probabilities)

# Display results
print("Maximum Return Portfolio Stability Analysis:")
print("=" * 80)
print(f"\nResampling Results (n={num_resamples}):")
print("\nAsset with Highest Return Probability:")
for ticker in tickers:
    count = highest_return_counts[ticker]
    prob = count / num_resamples
    print(f"  {ticker}: {count:4d} times ({prob*100:6.2f}%)")
print("\n" + "=" * 80)
print("\nDataFrame:")
print(max_return_prob_df)

Maximum Return Portfolio Stability Analysis:

Resampling Results (n=1000):

Asset with Highest Return Probability:
  AAPL:   91 times (  9.10%)
  SPY:    8 times (  0.80%)
  TSLA:  373 times ( 37.30%)
  QQQ:   30 times (  3.00%)
  MSFT:   71 times (  7.10%)
  NVDA:  427 times ( 42.70%)


DataFrame:
  asset  probability
0  AAPL        0.091
1   SPY        0.008
2  TSLA        0.373
3   QQQ        0.030
4  MSFT        0.071
5  NVDA        0.427


Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [37]:
# Save the max return probabilities to a TSV file
max_return_prob_path = 'max_return_probabilities.tsv'
max_return_prob_df.to_csv(max_return_prob_path, sep='\t', index=False)
print(f"Max return probabilities saved to {max_return_prob_path}\n")

# Display the final dataframe
print("Max Return Probabilities:")
print(max_return_prob_df)

Max return probabilities saved to max_return_probabilities.tsv

Max Return Probabilities:
  asset  probability
0  AAPL        0.091
1   SPY        0.008
2  TSLA        0.373
3   QQQ        0.030
4  MSFT        0.071
5  NVDA        0.427


Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.